# Kaffeequalität: EDA und Modellierung

Ein ausgearbeitetes Beispiel für den Machine-Learning-Workflow. Wir verwenden den Datensatz zur Qualität von Arabica-Kaffee, um die Gesamtbewertung beim Cupping (der professionellen Verkostung) jedes Kaffees aus seinen gemessenen Eigenschaften vorherzusagen, und prüfen dann, wie gut das Modell abschneidet. Nutze dieses Notebook als Vorlage für die Struktur deines eigenen Projekts.

## Der Machine-Learning-Workflow

### Das Geschäftsziel festlegen

Der Stakeholder ist ein **Einkäufer für Rohkaffee bei einem Kaffee-Importeur**. Jede Saison prüft er viele Arabica-Partien und möchte seine Zeit und sein Budget für die vielversprechendsten einsetzen.

Wir schätzen den **Cupping-Score** (Gesamtbewertung) eines Kaffees aus Eigenschaften, die *vor* einer vollständigen Verkostung bekannt sind. So kann der Einkäufer eine Vorauswahl der Partien treffen, die er verkosten will, Preisverhandlungen untermauern und vermeiden, Cupping-Sitzungen für Kaffees geringer Qualität zu verschwenden.

Der Score hängt von vielen Eigenschaften gleichzeitig ab. Ein Modell liefert daher eine schnellere und einheitlichere Schätzung als eine Faustregel von Hand.

Die Schätzung sollte genau genug sein, um in der Praxis nützlich zu sein, zum Beispiel auf ein paar Punkte genau auf der Skala von 0 bis 100.

In Data-Science-Begriffen ist das ein **Regressions**problem: Die Zielvariable `total_cup_points` ist eine stetige Zahl, und wir messen den Fehler mit dem **Root Mean Squared Error (RMSE)**. Er hat dieselbe Einheit wie der Score (Punkte).

### Die Daten holen

Wir laden den bereinigten Arabica-Datensatz. Er enthält sowohl die Eigenschaften der Kaffees als auch den Cupping-Gesamtscore (`Total.Cup.Points`), den wir als Zielvariable verwenden.

In [ ]:
import pandas as pd

# Arabica-Kaffeedaten (bereinigt)
URL = "https://raw.githubusercontent.com/jldbc/coffee-quality-database/master/data/arabica_data_cleaned.csv"
coffee = pd.read_csv(URL)

Schauen wir uns die Daten kurz an, um zu sehen, womit wir arbeiten.

In [ ]:
coffee.head()

### Erster Blick auf die Daten

Eine kurze Strukturprüfung zeigt die Datentypen der Spalten und wo Werte fehlen.

In [ ]:
coffee.info()

Der Datensatz enthält 1311 Arabica-Kaffees, beschrieben durch 44 Spalten: eine Mischung aus Text (Herkunft, Farm, Aufbereitungsmethode) und Zahlen (die Cupping-Scores, Anbauhöhe, Anzahl der Defekte). Mehrere Spalten haben fehlende Werte, und die Spalten zur Anbauhöhe enthalten außerdem einige unmögliche Einträge. All das bereinigen wir als Nächstes.

## Die Daten bereinigen

Diese Schritte sind alle **sicher vor Data Leakage**, darum machen wir sie vor dem Split: Spaltennamen aufräumen, Spalten und Zeilen entfernen, die wir nicht verwenden können, offensichtliche Eingabefehler korrigieren und Features entfernen, die die Antwort verraten würden. Nur einen Schritt verschieben wir auf nach dem Split: die Imputation, weil ihre Füllwerte aus den Daten gelernt werden.

In [ ]:
# Spaltennamen aufräumen: kleingeschrieben, mit Unterstrichen statt Punkten und Leerzeichen
coffee.columns = (
    coffee.columns.str.strip()
    .str.replace(r"[^0-9a-zA-Z]+", "_", regex=True)
    .str.strip("_")
    .str.lower()
)
coffee.columns.tolist()

Die unordentlichen Namen (zum Beispiel `Category.One.Defects` und `Total.Cup.Points`) werden zu ordentlichem snake_case. Das ist reines Umbenennen und daher immer sicher.

In [ ]:
# Spalten entfernen, bei denen mehr als die Hälfte der Werte fehlt
missing_fraction = coffee.isna().mean().sort_values(ascending=False)
high_missing = missing_fraction[missing_fraction > 0.5].index
print("Entferne Spalten mit mehr als 50 % fehlenden Werten:", list(high_missing))
coffee = coffee.drop(columns=high_missing)

Nur bei `lot_number` fehlt mehr als die Hälfte der Werte, darum entfernen wir diese Spalte.

In [ ]:
# Eingabefehler bei der Anbauhöhe korrigieren: 10000 Meter oder mehr sind unmöglich
# (so hoch wächst kein Kaffee), es ist also fast sicher ein Tippfehler um eine Zehnerpotenz.
# Solche Werte so oft durch 10 teilen, bis sie in einem glaubwürdigen Bereich liegen.
altitude_cols = ["altitude_low_meters", "altitude_high_meters", "altitude_mean_meters"]
print("Höhenwerte ab 10000 vor der Korrektur:", int((coffee[altitude_cols] >= 10000).sum().sum()))
for col in altitude_cols:
    while (coffee[col] >= 10000).any():
        coffee[col] = coffee[col].where(coffee[col] < 10000, coffee[col] / 10)
print("Größte Höhe nach der Korrektur:", round(coffee["altitude_mean_meters"].max(), 1))

Die Spalten zur Anbauhöhe enthalten unmögliche Werte wie 190164 Meter. Das sehen nach Tippfehlern um Zehnerpotenzen aus. Darum teilen wir jede Höhe ab 10000 so oft durch 10, bis sie glaubwürdig ist. Das ist eine feste Regel pro Zeile und damit sicher vor Data Leakage. Sie korrigiert 12 Werte in den drei Höhenspalten, und die größte Höhe sinkt auf etwa 4287 Meter.

In [ ]:
# Ein Kaffee hat einen Gesamtscore von 0, das ist unmöglich. Diese Zeile entfernen.
print("Zeilen mit einem Gesamtscore von 0:", int((coffee["total_cup_points"] == 0).sum()))
coffee = coffee[coffee["total_cup_points"] > 0]

Alle anderen Kaffees liegen zwischen etwa 59 und 91 Punkten, ein Score von 0 ist also eindeutig ein Eingabefehler. Wir entfernen diese eine Zeile, es bleiben 1310 Kaffees.

In [ ]:
# Fachwissen: Der Gesamtscore ist die SUMME der einzelnen Cupping-Scores.
# Das bestätigen und diese Einzelscores dann als Leakage behandeln.
component_scores = [
    "aroma", "flavor", "aftertaste", "acidity", "body",
    "balance", "uniformity", "clean_cup", "sweetness", "cupper_points",
]
sum_corr = coffee[component_scores].sum(axis=1).corr(coffee["total_cup_points"])
print("Korrelation der Summe der Einzelscores mit dem Gesamtscore:", round(sum_corr, 3))

Die Einzelscores ergeben zusammen die Gesamtbewertung (Korrelation 1.00). Damit sind sie **Target Leakage**: Würden wir sie dem Modell geben, bekäme es einfach die Antwort, zerlegt in zehn Teile. Sie sind außerdem für unser Ziel unrealistisch, denn der Einkäufer will eine Schätzung *vor* der Verkostung, wenn es diese Cupping-Scores noch gar nicht gibt.

Darum entfernen wir die Einzelscores jetzt und sagen die Qualität stattdessen aus wirklich unabhängigen Eigenschaften vorher.

In [ ]:
# Die Zielvariable ist der Cupping-Gesamtscore
y = coffee["total_cup_points"]

# Features: numerische Spalten, ohne Zeilenindex, Zielvariable und die verräterischen Einzelscores
X = coffee.select_dtypes("number").drop(
    columns=["unnamed_0", "total_cup_points"] + component_scores
)
X.columns.tolist()

Übrig bleiben acht unabhängige Eigenschaften: die Anzahl der Säcke, die Feuchtigkeit, die beiden Defekt-Zählungen, die Quakers (unreife, hell bleibende Bohnen) und die drei Höhenspalten. Das sind Dinge, die ein Einkäufer über eine Partie wissen kann, ohne sie zu verkosten.

## Train-Test-Split

Wir halten einen Teil der Daten zurück, damit wir das Modell an Kaffees beurteilen können, die es nie gesehen hat. Ab hier wird alles, was wir *fitten* (zum Beispiel Werte für die Imputation), nur auf dem Trainingsset gelernt.

Wir verwenden einen einzigen Train-Test-Split (80 % Training, 20 % Test). Ein größeres Projekt würde zusätzlich ein Validierungsset behalten oder Cross-Validation nutzen, um das Modell zu optimieren, ohne das Testset anzurühren.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

Die 1310 Kaffees werden in 1048 zum Trainieren und 262 zum Testen aufgeteilt.

## Die Trainingsdaten erkunden

Jetzt erkunden wir das **Trainingsset**, um die Features zu verstehen, bevor wir irgendetwas fitten.

In [ ]:
X_train.info()

Es gibt acht numerische Features. Die meisten sind vollständig, aber bei `quakers` fehlt ein Wert, und die drei Höhenspalten fehlen bei 187 der 1048 Trainings-Kaffees. Diese Lücken imputieren wir nach dem Split.

In [ ]:
# Wie korreliert jedes verbleibende Feature mit der Zielvariable?
X_train.corrwith(y_train).sort_values(ascending=False)

Keine der unabhängigen Eigenschaften hängt stark mit dem Score zusammen. Das deutlichste Signal hat `category_two_defects` (etwa -0.28: mehr Defekte, niedrigerer Score), gefolgt von Feuchtigkeit und Anbauhöhe, alle schwach.

## Feature Engineering

Wir füllen jeden fehlenden Wert (`quakers` und die drei Höhenspalten) mit dem Mittelwert seiner Spalte im **Trainingsset** und verwenden genau diese Werte später für das Testset.

In [ ]:
# Die Füllwerte nur auf dem Trainingsset lernen
train_means = X_train.mean()
X_train = X_train.fillna(train_means)
train_means

`train_means` enthält einen Wert pro Spalte. Genau diese Werte verwenden wir bei der Bewertung auch für das Testset. Das sorgt dafür, dass die Bewertung ehrlich bleibt.

## Baseline-Modell

Bevor wir ein Modell trainieren, legen wir eine **Baseline** fest: die einfachste Regel, die eine Vorhersage macht.

Bei einer Regression wie unserer ist eine übliche Faustregel, die Features zu ignorieren und immer den **durchschnittlichen Trainings-Score** vorherzusagen. Wenn unser Modell das nicht schlägt, hat es nichts Brauchbares gelernt.

In [ ]:
# Heuristische Baseline: Features ignorieren, immer den mittleren Trainings-Score vorhersagen
from sklearn.metrics import root_mean_squared_error

baseline_prediction = y_train.mean()
baseline_rmse = root_mean_squared_error(y_test, [baseline_prediction] * len(y_test))

print(f"Die Baseline sagt immer vorher: {baseline_prediction:.2f}")
print(f"Test-RMSE der Baseline: {baseline_rmse:.2f}")

Die Baseline sagt immer etwa 82.2 vorher (den mittleren Trainings-Score) und erreicht einen Test-RMSE von etwa 2.66. Sie liegt also im Schnitt um etwa 2 bis 3 Punkte neben dem wahren Score. Diesen Wert muss unser Modell schlagen.

## Das Modell trainieren

Wir trainieren ein lineares Regressionsmodell: schnell, leicht zu lesen und ein sinnvoller Schritt über die Baseline hinaus. Zuerst legen wir das Testset beiseite und speichern eine Kopie, dann fitten wir das Modell und prüfen den Fehler auf den Trainingsdaten.

In [ ]:
# Das Testset beiseitelegen und eine Kopie speichern. Wir rühren es bis zur finalen Bewertung nicht an.
X_test.to_csv("data/X_test.csv")
y_test.to_csv("data/y_test.csv")

In [ ]:
from sklearn.linear_model import LinearRegression
reg = LinearRegression().fit(X_train, y_train)

In [ ]:
y_train_pred = reg.predict(X_train)
rmse = root_mean_squared_error(y_train, y_train_pred)
print(rmse)

Der Trainings-RMSE liegt bei etwa 2.54 Punkten, nah an der Baseline. Das Modell lernt die Daten nicht auswendig, was nicht überrascht, da die Features nur schwach mit dem Score zusammenhängen.

## Das Modell bewerten

Wir wenden denselben gefitteten Schritt auf das Testset an: Imputation mit den Mittelwerten aus dem **Trainingsset** (nie aus den Testdaten neu berechnet). Die Bereinigung und die Korrektur der Anbauhöhe sind schon vor dem Split passiert, das Testset hat sie also auch. Dann bewerten wir das Modell an diesen ungesehenen Kaffees.

In [ ]:
# Den gefitteten Schritt (Imputation) mit den Trainings-Mittelwerten auf das Testset anwenden
X_test = X_test.fillna(train_means)

In [ ]:
y_test_pred = reg.predict(X_test)
rmse = root_mean_squared_error(y_test, y_test_pred)
print(rmse)

Der Test-RMSE liegt bei etwa 2.46, knapp unter den 2.66 der Baseline. Das Modell ist nur ein wenig besser, als immer den Durchschnitt zu raten: Die Eigenschaften, die ein Einkäufer vor der Verkostung kennen kann (Anbauhöhe, Defekte, Feuchtigkeit, Anzahl der Säcke), enthalten nur ein schwaches Signal über den endgültigen Cupping-Score.

Die naheliegenden nächsten Schritte im Workflow sind: aussagekräftigere Features oder andere Modelle ausprobieren, das beste auf Validierungsdaten oder per Cross-Validation auswählen, Hyperparameter-Tuning, Fehleranalyse, einen finalen Test-Score berichten und das Modell dann in Betrieb nehmen (Deployment) und überwachen (Monitoring).

In [ ]:
# Das beste Modell für später in einer Datei speichern
import joblib
joblib.dump(reg, "models/linear_regression_model.joblib")

In [ ]:
# Mit dem Modell den Score eines neuen Kaffees mit diesen Eigenschaften vorhersagen:
 
new_coffee = pd.DataFrame({
    
    "number_of_bags" :[1] ,
    "moisture": [10.5] , 
    "category_one_defects": [0],
    "quakers": [0],
    "category_two_defects": [1], 
    "altitude_low_meters": [1200], 
    "altitude_high_meters": [1500],
    "altitude_mean_meters": [1350]
})
new_coffee = new_coffee.fillna(train_means)  # dieselbe Imputation wie bei den Trainingsdaten anwenden

In [ ]:
X_train.columns

In [ ]:
# Das Modell aus der Datei laden und damit den Score des neuen Kaffees vorhersagen
loaded_model = joblib.load("models/linear_regression_model.joblib")
predicted_score = loaded_model.predict(new_coffee)[0]
print(f"Vorhergesagter Cupping-Score für den neuen Kaffee: {predicted_score:.2f}")

## Quellen und weiterführende Literatur

Die externen Ressourcen sind auf Englisch.

- [**LinearRegression (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html): das Modell, das in diesem Notebook trainiert wird.
- [**train_test_split (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html): wie das zurückgehaltene Testset erstellt wird.
- [**root_mean_squared_error (scikit-learn)**](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.root_mean_squared_error.html): die Metrik, mit der Modell und Baseline bewertet werden.
- [**Common pitfalls and recommended practices (scikit-learn)**](https://scikit-learn.org/stable/common_pitfalls.html): wie Data Leakage entsteht und wie man es vermeidet.
- [**Coffee Quality Database**](https://github.com/jldbc/coffee-quality-database): die Quelle des hier verwendeten Datensatzes.